In [1]:
import os
from elasticsearch import Elasticsearch
import json
import pandas as pd

from extraction_prompts import (
    EXTRACT_RELATION_TRIPLETS_PROMPT,
    REFORMULATE_RELATION_TRIPLET,
    VALIDATE_TRIPLET_USEFULNESS,

)
from utils import ollama_request

ES_HOST = os.getenv("ES_HOST", "http://localhost:9200")
INDEX_NAME = os.getenv("INDEX_NAME")
FRAGMENT_INDEX_NAME = f"{os.getenv('INDEX_NAME')}_fragments"
es_client = Elasticsearch('http://localhost:9200')


/home/zbrzeznyg/miniconda3/envs/masters/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def create_relations(es_client, fragment_index_name, step=2, reformulate_predicate=True, speech_id_start=1, speech_id_end=10):
    try:
        final_triplets = pd.read_csv("data/triplets.csv")
    except:
        final_triplets = pd.DataFrame(columns=["speech_id", "start", "end", "date", "subject", "predicate", "object"])
    for speech_id in range(speech_id_start, speech_id_end + 1):
        retrieve_attempts = 0
        fragments_for_text = es_client.search(
            index=fragment_index_name,
            query={
                "match": {
                    "speech_id": speech_id
                }
            },
            size=10000
        )
        hits = fragments_for_text['hits']['hits']
        if len(hits) == 0:
            continue
        print(f"Processing speech_id: {speech_id}, number of fragments: {len(hits)}")
        new_triplets = pd.DataFrame(columns=["speech_id", "start", "end", "date", "subject", "predicate", "object"])
        for fragment_id in range(0, len(hits), step):
            fragment_group = hits[fragment_id:fragment_id + step]
            fragment_text = " ".join(
                hit["_source"]["text"]
                for hit in fragment_group
                if hit["_source"].get("text")
            )
            jsoned_triplets = None
            for attempt in range(2):
                triplets = ollama_request(
                    prompt=EXTRACT_RELATION_TRIPLETS_PROMPT.format(fragment=fragment_text, speech_author=fragment_group[0]["_source"].get("author", "Unknown")),
                    is_stream=False
                ).replace("```json", "").replace("```", "").replace("*", "").strip()

                try:
                    jsoned_triplets = json.loads(triplets)["triplets"]
                    break
                except Exception as e:
                    if attempt != 0:
                        continue
                    # else:
                    #     print(f"JSON decoding error for speech_id {speech_id}, fragment_id {fragment_id}, Retrying once with the same prompt...")

            if jsoned_triplets is None:
                continue

            for triplet in jsoned_triplets:
                if triplet is None or not isinstance(triplet, dict):
                    continue
                subject = triplet.get("subject", None)
                predicate = triplet.get("predicate", None)
                object_ = triplet.get("object", None)

                if object_:
                    object_ = str(object_).strip()

                if any(not x for x in [subject, predicate, object_]):
                    continue

                reformulated_triplet = {}
                if reformulate_predicate:
                    reformulated_predicate = ollama_request(
                            prompt=REFORMULATE_RELATION_TRIPLET.format(subject=subject, predicate=predicate, object=object_),
                            is_stream=False
                    ).replace("```json", "").replace("```", "").replace("*", "").replace('<', "").replace('>', "").replace('.', "").strip()
                    # print(f"Changed {predicate} to: {reformulated_predicate}")
                    reformulated_triplet["predicate"] = reformulated_predicate

                reformulated_triplet = {
                    "subject": reformulated_triplet.get("subject", subject).split("(")[0].strip(),
                    "predicate": reformulated_triplet.get("predicate", predicate),
                    "object": reformulated_triplet.get("object", object_).split("(")[0].strip()
                }
                
                is_valid = ollama_request(
                    prompt=VALIDATE_TRIPLET_USEFULNESS.format(
                        subject=reformulated_triplet.get("subject", subject),
                        predicate=reformulated_triplet.get("predicate", predicate),
                        object=reformulated_triplet.get("object", object_)
                    ),
                    is_stream=False
                )
                if is_valid.strip().lower().replace('"', "").replace("'", "") != "useful":
                    continue
                new_triplets = pd.concat([new_triplets, pd.DataFrame([{
                    "speech_id": speech_id,
                    "start": fragment_group[0]["_source"]["chunk_start"],
                    "end": fragment_group[-1]["_source"]["chunk_end"],
                    "date": fragment_group[0]["_source"]["date"],
                    "subject": reformulated_triplet.get("subject", subject).lower().replace('<', ' ').replace('>', ' ').replace('.', ' ').replace("'", '').strip(),
                    "predicate": reformulated_triplet.get("predicate", predicate).replace('<', ' ').replace('>', ' ').replace('.', ' ').replace("'", '').lower().strip(),
                    "object": reformulated_triplet.get("object", object_).lower().replace('<', ' ').replace('>', ' ').replace('.', ' ').replace("'", '').strip()
                }])], ignore_index=True)
        final_triplets = pd.concat([final_triplets[final_triplets["speech_id"] != speech_id], new_triplets], ignore_index=True)
        final_triplets.to_csv("data/triplets.csv", index=False)

In [ ]:
triplets = create_relations(es_client, FRAGMENT_INDEX_NAME, step=1, reformulate_predicate=True, speech_id_start=5715, speech_id_end=10000)

Processing speech_id: 4587, number of fragments: 38
Processing speech_id: 4588, number of fragments: 10
Processing speech_id: 4589, number of fragments: 28
Processing speech_id: 4590, number of fragments: 8
Processing speech_id: 4591, number of fragments: 22
Processing speech_id: 4592, number of fragments: 27
Processing speech_id: 4593, number of fragments: 13
Processing speech_id: 4594, number of fragments: 22
Processing speech_id: 4595, number of fragments: 65
Processing speech_id: 4596, number of fragments: 16
Processing speech_id: 4597, number of fragments: 8
Processing speech_id: 4598, number of fragments: 6
Processing speech_id: 4599, number of fragments: 6
Processing speech_id: 4600, number of fragments: 14
Processing speech_id: 4601, number of fragments: 6
Processing speech_id: 4602, number of fragments: 5
Processing speech_id: 4603, number of fragments: 15
Processing speech_id: 4604, number of fragments: 7
Processing speech_id: 4605, number of fragments: 24
Processing speech_i

KeyboardInterrupt: 